In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

# 1. Считываем датасет из файла train.csv
df = pd.read_csv('train.csv')

# 2. Местрика
print("\n Для задачи предсказания выживания пассажиров Титаника наиболее релевантной метрикой является F1-score, т.к. F1-score представляет собой среднее между точностью (precision) и полнотой (recall), что делает его более надежным в случае, если классы выживших и погибших не полностью сбалансированы, а так же в задачах классификации, важно учитывать как минимизацию ложных срабатываний (точность), так и максимизацию нахождения всех реальных случаев (полнота). F1-score учитывает оба этих аспекта.")

# 3. Данные
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
target = 'Survived'

# Создадим копию датафрейма для обработки
df_processed = df[features + [target]].copy()

# Заполним пропущенные значения возраста медианой
df_processed['Age'].fillna(df_processed['Age'].median(), inplace=True)

# Закодируем признак 'Sex'
label_encoder = LabelEncoder()
df_processed['Sex'] = label_encoder.fit_transform(df_processed['Sex']) # male: 1, female: 0

# Проверим, есть ли пропущенные значения в 'Fare'
if df_processed['Fare'].isnull().any():
    df_processed['Fare'].fillna(df_processed['Fare'].median(), inplace=True)

print("\n Подготовка данных ")
print(f"Признаки для моделирования: {features}")
print(f"Целевая переменная: {target}")
print(f"Пропущенные значения в признаках:")
print(df_processed[features].isnull().sum())

# Разделим данные на признаки (X) и целевую переменную (y)
X = df_processed[features]
y = df_processed[target]

# Разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13, stratify=y)

print(f"\n Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

# Масштабируем признаки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Построение бейзлайна и ML-модели

# Бейзлайн 1: Предсказание самого частого класса (majority class)
baseline_majority_pred = np.full_like(y_test, fill_value=y_train.mode()[0])
baseline_majority_accuracy = accuracy_score(y_test, baseline_majority_pred)
baseline_majority_f1 = f1_score(y_test, baseline_majority_pred)
baseline_majority_roc_auc = roc_auc_score(y_test, baseline_majority_pred)

print("\n Результаты бейзлайнов ")
print("Бейзлайн 1 (Предсказание самого частого класса):")
print(f"  Accuracy: {baseline_majority_accuracy:.4f}")
print(f"  F1-score: {baseline_majority_f1:.4f}")
print(f"  ROC-AUC:  {baseline_majority_roc_auc:.4f}")

# Бейзлайн 2: Случайное предсказание с учетом распределения классов

# ML-модель 1: Логистическая регрессия
print("\n Результаты ML-моделей ")

# Логистическая регрессия
lr_model = LogisticRegression(random_state=13, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1] # Вероятности для класса 1

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_roc_auc = roc_auc_score(y_test, lr_pred_proba)

print("Модель 1 (Логистическая регрессия):")
print(f"  Accuracy:  {lr_accuracy:.4f}")
print(f"  Precision: {lr_precision:.4f}")
print(f"  Recall:    {lr_recall:.4f}")
print(f"  F1-score:  {lr_f1:.4f}")
print(f"  ROC-AUC:   {lr_roc_auc:.4f}")

# ML-модель 2: Random Forest (для сравнения)
rf_model = RandomForestClassifier(n_estimators=100, random_state=13)
rf_model.fit(X_train, y_train) # Random Forest не требует масштабирования
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_pred_proba)

print("\n Модель 2 (Random Forest):")
print(f"  Accuracy:  {rf_accuracy:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall:    {rf_recall:.4f}")
print(f"  F1-score:  {rf_f1:.4f}")
print(f"  ROC-AUC:   {rf_roc_auc:.4f}")


 Для задачи предсказания выживания пассажиров Титаника наиболее релевантной метрикой является F1-score, т.к. F1-score представляет собой среднее между точностью (precision) и полнотой (recall), что делает его более надежным в случае, если классы выживших и погибших не полностью сбалансированы, а так же в задачах классификации, важно учитывать как минимизацию ложных срабатываний (точность), так и максимизацию нахождения всех реальных случаев (полнота). F1-score учитывает оба этих аспекта.

 Подготовка данных 
Признаки для моделирования: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
Целевая переменная: Survived
Пропущенные значения в признаках:
Pclass    0
Sex       0
Age       0
SibSp     0
Parch     0
Fare      0
dtype: int64

 Размер обучающей выборки: (712, 6)
Размер тестовой выборки: (179, 6)

 Результаты бейзлайнов 
Бейзлайн 1 (Предсказание самого частого класса):
  Accuracy: 0.6145
  F1-score: 0.0000
  ROC-AUC:  0.5000

 Результаты ML-моделей 
Модель 1 (Логистическая регресс

/tmp/ipython-input-2-3199040577.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_processed['Age'].fillna(df_processed['Age'].median(), inplace=True)



 Модель 2 (Random Forest):
  Accuracy:  0.8212
  Precision: 0.7467
  Recall:    0.8116
  F1-score:  0.7778
  ROC-AUC:   0.8773
